# 01 · Timestamp discipline

The failure mode of nearly all text-signal research is treating a document as
usable at a bar it could not have been acted on. It is rarely a dramatic bug —
it is an off-by-one in *when you were allowed to know something*, and it
inflates results quietly.

This notebook walks the machinery in `app/text_signals/timestamps.py` that makes
the relevant instants explicit and refuses to let them be conflated.

**What to take from it:** availability and executability are different
questions, and a pipeline that answers only one of them is not safe.

In [ ]:
# Put `backend/` on sys.path so `app.*` imports work regardless of where
# Jupyter was launched from. Walks up until it finds the backend package.
import sys, pathlib

here = pathlib.Path.cwd()
for candidate in (here, *here.parents):
    if (candidate / "backend" / "app").is_dir():
        sys.path.insert(0, str(candidate / "backend"))
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("could not locate backend/ — run from inside the repo")

FIXTURES = REPO_ROOT / "backend" / "tests" / "fixtures" / "edgar"
print("repo root :", REPO_ROOT)
print("fixtures  :", FIXTURES, "(exists)" if FIXTURES.is_dir() else "(MISSING)")

## The five instants

```
event_time                  when the underlying thing happened
publish_time                when the document became public
ingest_time                 when this pipeline actually received it
information_available_time  = max(publish_time + buffer, ingest_time)   [derived]
execution_time              = next session open at/after available      [derived]
```

The last two are **derived**. They cannot be supplied by a caller — that is
enforced, not documented, and the next cell shows why it matters.

In [ ]:
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

from app.text_signals.timestamps import (
    DEFAULT_DISSEMINATION_BUFFER,
    IngestSource,
    StaticHolidayCalendar,
    TextRecord,
)

ET = ZoneInfo("America/New_York")

# Illustrative calendar only. `approved_for_research` is a claim about
# verification, so in real use this comes from a maintained source — the
# module deliberately refuses test-grade calendars on research paths.
CALENDAR = StaticHolidayCalendar(
    name="demo-nyse-2026",
    holidays=frozenset(),
    start=datetime(2026, 1, 1).date(),
    end=datetime(2026, 12, 31).date(),
    approved_for_research=True,
)

# A 10-K accepted after the close on a Friday.
filing = TextRecord(
    doc_id="0000320193-26-000001",
    symbol="AAPL",
    publish_time=datetime(2026, 8, 14, 18, 5, tzinfo=ET),   # Friday, after close
    ingest_time=datetime(2026, 8, 14, 18, 20, tzinfo=ET),
    ingest_time_source=IngestSource.OBSERVED,
    source="edgar:10-K",
)

resolved = filing.resolve(CALENDAR)
print("publish_time               :", resolved.publish_time)
print("ingest_time                :", resolved.ingest_time)
print("dissemination buffer       :", DEFAULT_DISSEMINATION_BUFFER)
print("information_available_time :", resolved.information_available_time)
print("execution_time             :", resolved.execution_time)
print()
print("publish -> execution gap   :", resolved.publish_to_execution_gap)

Note the gap. The filing was *public* Friday evening and *available* half an
hour later — but the first moment a position could be taken is Monday's open.
Anything that scores this signal against Friday's close is reading the future.

In [ ]:
# Derived instants are not constructor arguments. Supplying one is a TypeError,
# not a value that quietly wins over the derivation.
try:
    TextRecord(
        doc_id="x",
        symbol="AAPL",
        publish_time=datetime(2026, 8, 14, 18, 5, tzinfo=ET),
        ingest_time=datetime(2026, 8, 14, 18, 20, tzinfo=ET),
        ingest_time_source=IngestSource.OBSERVED,
        information_available_time=datetime(2026, 8, 14, 9, 30, tzinfo=ET),  # wishful
    )
except TypeError as exc:
    print("TypeError (as designed):", exc)

## `ingest_time` participates

A filing this pipeline had not yet received was not available *to this
pipeline*, however public it was. When late receipt — not dissemination — sets
availability, the record is **ingest-bound**, and a corpus full of those is one
whose availability claim rests on backfill assumptions.

In [ ]:
late = TextRecord(
    doc_id="late-arrival",
    symbol="MSFT",
    publish_time=datetime(2026, 8, 14, 10, 0, tzinfo=ET),
    ingest_time=datetime(2026, 8, 17, 8, 0, tzinfo=ET),   # received 3 days later
    ingest_time_source=IngestSource.SIMULATED,
    source="edgar:10-K",
).resolve(CALENDAR)

print("published            :", late.publish_time)
print("received             :", late.ingest_time)
print("available            :", late.information_available_time)
print("ingest_binding       :", late.ingest_binding, "<- receipt, not dissemination, set availability")
print("execution            :", late.execution_time)

## Calendars fail closed

Two separate guards, both of which exist because the alternative is silent
drift:

1. a **test-grade** calendar is rejected on research paths;
2. a real calendar asked about a date **outside its verified range** raises
   instead of extrapolating.

In [ ]:
from app.text_signals.timestamps import (
    CalendarCoverageError,
    UnapprovedCalendarError,
    WeekdayCalendar,
)

# 1. Test-grade calendar on a research path.
try:
    filing.resolve(WeekdayCalendar())
except UnapprovedCalendarError as exc:
    print("UnapprovedCalendarError:", exc)

# 2. Outside verified coverage.
try:
    CALENDAR.is_session(datetime(2031, 3, 3).date())
except CalendarCoverageError as exc:
    print("CalendarCoverageError  :", exc)

## The look-ahead guard

`assert_no_lookahead` belongs at the top of every feature builder. Equality is
admissible — if information is available *at* the computation instant, it
exists then. Whether the resulting position is executable is the separate
question `execution_time` answers.

In [ ]:
from app.text_signals.timestamps import LookaheadError, assert_no_lookahead, usable_at

records = [resolved, late]

# Computing a feature on Monday morning: both filings are legitimately readable.
monday = datetime(2026, 8, 17, 9, 30, tzinfo=ET)
assert_no_lookahead(records, monday)
print("Monday 09:30 — both readable, no look-ahead")

# Computing on Friday afternoon: the after-close filing is not knowable yet.
friday_pm = datetime(2026, 8, 14, 15, 0, tzinfo=ET)
try:
    assert_no_lookahead(records, friday_pm)
except LookaheadError as exc:
    print("LookaheadError:", exc)

print()
print("usable_at(Friday 15:00):", [r.doc_id for r in usable_at(records, friday_pm)])
print("usable_at(Monday 09:30):", [r.doc_id for r in usable_at(records, monday)])

## What belongs in an evidence package

The gap distribution is a reported metric, not an implementation detail.
Concentrated near the buffer means documents arrive during session hours; a long
right tail means weekend and after-hours filings dominate — and those two cases
support different claims about what the signal can be.

In [ ]:
from app.text_signals.timestamps import gap_summary

for key, value in gap_summary(records).items():
    print(f"{key:24} {value}")

`simulated_ingest_share` and `ingest_bound_share` are the honesty numbers. A
fully simulated study is not disqualified — it is *described*, and the figure
belongs beside the result rather than in a footnote.